In [ ]:
# -*- coding: utf-8 -*-
"""
mdhtml.py - render the shipped Markdown documents as a self-contained page.

The manual is a long reference with a table of contents, and a table of
contents is only useful if its links work. A Tk Text widget can be taught to
jump to a mark, but it will not give you Ctrl+F, zoom, print, or back after
following a link, and it lays proportional-font tables out badly enough that
the in-window reader gives up on them entirely.

A browser does all of that already. So the documents are converted here and
opened as HTML, and the reader window stays as the fallback for a machine with
no browser to open.

Two constraints shape this:

  * SELF-CONTAINED. The CSS is inlined and nothing is fetched. The program is
    used on managed and offline machines, and a page that needs the network to
    look right is a page that sometimes does not.
  * THE SAME SLUGS the documents already link to. HELP.md writes its contents
    as [Ground Truth](#ground-truth); an anchor generated by any other rule
    silently fails to match, which is the exact fault this replaces.

Deliberately not a complete Markdown implementation - it covers what these
documents use, and is checked against them rather than against a specification.
"""

import html
import re

__all__ = ['slug', 'to_html', 'render_page']


def slug(heading):
    """The anchor for a heading, spelled the way the documents link to it."""
    text = re.sub(r'`([^`]*)`', r'\1', heading)
    text = re.sub(r'\*\*([^*]*)\*\*', r'\1', text)
    text = re.sub(r'\[([^\]]*)\]\([^)]*\)', r'\1', text)
    text = re.sub(r'[^a-z0-9 -]', '', text.lower())
    return text.strip().replace(' ', '-')


def _local_href(url):
    """A link to a sibling document points at the page made from that document.

    HELP.md says [INSTALL.md](INSTALL.md), which is right on GitHub and dead in
    a browser: the folder these pages are written to holds .html, not .md. The
    fragment is carried across, so [HELP.md#citation] still lands on the
    section. Anything with a scheme is somebody else's URL and is left alone.
    """
    m = re.match(r'^([^#?:]+)\.md(#.*)?$', url)
    return f'{m.group(1)}.html{m.group(2) or ""}' if m else url


def _inline(text):
    """Emphasis, code, and links, inside one line.

    Code spans are lifted out first and put back last. What is inside them is
    literal: the manual writes `*_mean_relevancy.db` and `score_*` in the same
    sentence, and left in place the asterisk ending one span paired with the
    asterisk opening the next, italicising the text between two column names.
    """
    out = html.escape(text, quote=False)
    kept = []

    def stash(m):
        kept.append(f'<code>{m.group(1)}</code>')
        return f'\x00{len(kept) - 1}\x00'

    out = re.sub(r'`([^`]+)`', stash, out)
    out = re.sub(r'\*\*([^*]+)\*\*', r'<strong>\1</strong>', out)
    out = re.sub(r'(?<![\w*])\*([^*\n]+)\*(?![\w*])', r'<em>\1</em>', out)
    # Links after emphasis: the label may itself carry the markup above.
    out = re.sub(r'\[([^\]]+)\]\(([^)\s]+)\)',
                 lambda m: f'<a href='
                           f'"{html.escape(_local_href(m.group(2)), quote=True)}">'
                           f'{m.group(1)}</a>', out)
    # A bare URL in angle brackets, which the citations use.
    out = re.sub(r'&lt;(https?://[^&\s]+)&gt;',
                 lambda m: f'<a href="{m.group(1)}">{m.group(1)}</a>', out)
    return re.sub(r'\x00(\d+)\x00', lambda m: kept[int(m.group(1))], out)


def _table(rows):
    """A pipe table. The second row is the alignment rule, not data."""
    def cells(line):
        line = line.strip()
        if line.startswith('|'):
            line = line[1:]
        if line.endswith('|'):
            line = line[:-1]
        return [c.strip() for c in line.split('|')]

    head = cells(rows[0])
    aligns = []
    for spec in cells(rows[1]):
        left, right = spec.startswith(':'), spec.endswith(':')
        aligns.append('center' if left and right else 'right' if right else 'left')
    out = ['<table>', '<thead><tr>']
    for i, c in enumerate(head):
        a = aligns[i] if i < len(aligns) else 'left'
        out.append(f'<th style="text-align:{a}">{_inline(c)}</th>')
    out.append('</tr></thead><tbody>')
    for line in rows[2:]:
        out.append('<tr>')
        for i, c in enumerate(cells(line)):
            a = aligns[i] if i < len(aligns) else 'left'
            out.append(f'<td style="text-align:{a}">{_inline(c)}</td>')
        out.append('</tr>')
    out.append('</tbody></table>')
    return '\n'.join(out)


def to_html(text):
    """Markdown body -> HTML fragment. Headings carry ids matching slug()."""
    lines = text.replace('\r\n', '\n').split('\n')
    out, i = [], 0
    list_stack = []          # open list tags, innermost last

    def close_lists(to_depth=0):
        while len(list_stack) > to_depth:
            out.append(f'</{list_stack.pop()}>')

    while i < len(lines):
        raw = lines[i]
        line = raw.rstrip()
        stripped = line.strip()

        if stripped.startswith('```'):
            close_lists()
            i += 1
            block = []
            while i < len(lines) and not lines[i].strip().startswith('```'):
                block.append(html.escape(lines[i], quote=False))
                i += 1
            i += 1
            out.append('<pre><code>' + '\n'.join(block) + '</code></pre>')
            continue

        if not stripped:
            close_lists()
            i += 1
            continue

        if set(stripped) <= {'-', '*', '_'} and len(stripped) >= 3:
            close_lists()
            out.append('<hr>')
            i += 1
            continue

        m = re.match(r'^(#{1,6})\s+(.*)$', stripped)
        if m:
            close_lists()
            level, title = len(m.group(1)), m.group(2).strip()
            out.append(f'<h{level} id="{slug(title)}">{_inline(title)}</h{level}>')
            i += 1
            continue

        # A pipe table needs its alignment rule on the following line.
        if (stripped.startswith('|') and i + 1 < len(lines)
                and re.match(r'^\s*\|?[\s:|-]+\|[\s:|-]*$', lines[i + 1])):
            close_lists()
            rows = []
            while i < len(lines) and lines[i].strip().startswith('|'):
                rows.append(lines[i])
                i += 1
            out.append(_table(rows))
            continue

        if stripped.startswith('>'):
            close_lists()
            quote = []
            while i < len(lines) and lines[i].strip().startswith('>'):
                quote.append(lines[i].strip()[1:].strip())
                i += 1
            out.append('<blockquote>' + _inline(' '.join(quote)) + '</blockquote>')
            continue

        bullet = re.match(r'^(\s*)([-*+]|\d+[.)])\s+(.*)$', line)
        if bullet:
            indent, marker, body = bullet.groups()
            depth = len(indent) // 2 + 1
            tag = 'ol' if marker[0].isdigit() else 'ul'
            while len(list_stack) > depth:
                out.append(f'</{list_stack.pop()}>')
            if len(list_stack) < depth:
                while len(list_stack) < depth:
                    out.append(f'<{tag}>')
                    list_stack.append(tag)
            elif list_stack and list_stack[-1] != tag:
                out.append(f'</{list_stack.pop()}>')
                out.append(f'<{tag}>')
                list_stack.append(tag)
            out.append(f'<li>{_inline(body)}</li>')
            i += 1
            continue

        close_lists()
        para = []
        while i < len(lines) and lines[i].strip() and not re.match(
                r'^\s*(#{1,6}\s|```|\||>|[-*+]\s|\d+[.)]\s)', lines[i]):
            para.append(lines[i].strip())
            i += 1
        if para:
            out.append('<p>' + _inline(' '.join(para)) + '</p>')
        else:
            i += 1

    close_lists()
    return '\n'.join(out)


# Inlined so the page is correct with no network and no shared stylesheet.
# Follows the reader's theme rather than forcing one, because this is opened
# beside an application the user may be running in either.
_CSS = """
:root { color-scheme: light dark; }
* { box-sizing: border-box; }
body { margin: 0; background: #ffffff; color: #1b1b1b;
       font: 16px/1.65 -apple-system, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; }
.wrap { max-width: 52rem; margin: 0 auto; padding: 2.5rem 1.5rem 6rem; }
h1, h2, h3, h4 { line-height: 1.25; margin: 2rem 0 .6rem; scroll-margin-top: 1rem; }
h1 { font-size: 2rem; margin-top: .5rem; }
h2 { font-size: 1.45rem; border-bottom: 1px solid #e2e5e9; padding-bottom: .3rem; }
h3 { font-size: 1.15rem; }
h4 { font-size: 1rem; }
p, li { margin: .55rem 0; }
ul, ol { padding-left: 1.5rem; }
a { color: #0b5fa5; text-decoration: none; }
a:hover { text-decoration: underline; }
code { background: #f0f2f4; border-radius: 3px; padding: .1em .35em;
       font: .875em/1.5 ui-monospace, "Cascadia Mono", Consolas, monospace; }
pre { background: #f6f7f9; border: 1px solid #e2e5e9; border-radius: 6px;
      padding: .9rem 1rem; overflow-x: auto; }
pre code { background: none; padding: 0; font-size: .85rem; }
blockquote { margin: 1rem 0; padding: .5rem 1rem; border-left: 4px solid #c9ced4;
             background: #f8f9fa; color: #40474e; }
table { border-collapse: collapse; width: 100%; margin: 1rem 0;
        display: block; overflow-x: auto; }
th, td { border: 1px solid #dfe3e7; padding: .45rem .65rem; vertical-align: top; }
th { background: #f3f5f7; font-weight: 600; }
tr:nth-child(even) td { background: #fbfcfd; }
hr { border: 0; border-top: 1px solid #e2e5e9; margin: 2rem 0; }
.note { margin: 0 0 1.5rem; padding: .6rem .9rem; background: #f3f5f7;
        border: 1px solid #e2e5e9; border-radius: 6px; font-size: .9rem;
        color: #454c53; }
@media (prefers-color-scheme: dark) {
  body { background: #16181b; color: #e6e8ea; }
  h2 { border-bottom-color: #2c3138; }
  a { color: #6cb6ff; }
  code { background: #23262b; }
  pre { background: #1b1e22; border-color: #2c3138; }
  blockquote { background: #1b1e22; border-left-color: #3a414a; color: #b9bfc6; }
  th, td { border-color: #2c3138; }
  th { background: #21252a; }
  tr:nth-child(even) td { background: #191c20; }
  hr { border-top-color: #2c3138; }
  .note { background: #1b1e22; border-color: #2c3138; color: #b9bfc6; }
}
@media print { .note { display: none; } a { color: inherit; } }
"""


def render_page(text, title, source=None):
    """A complete, self-contained HTML document."""
    where = (f'<p class="note">Read-only copy generated from '
             f'<code>{html.escape(str(source))}</code>. Edit that file, not this '
             f'page.</p>' if source else '')
    return (
        '<!doctype html>\n<html lang="en">\n<head>\n'
        '<meta charset="utf-8">\n'
        '<meta name="viewport" content="width=device-width, initial-scale=1">\n'
        f'<title>{html.escape(title)}</title>\n'
        f'<style>{_CSS}</style>\n</head>\n<body>\n<div class="wrap">\n'
        f'{where}\n{to_html(text)}\n</div>\n</body>\n</html>\n')
